# Reproduce the headline finding

**HLA-B\*58:01 is preferentially lost under HLA-LOH.** This notebook recomputes that result from scratch, reading only the committed aggregated table `data/derived/msk50k_within_patient_LOH_bias.csv` (per-allele counts, no patient-level rows). No large downloads needed.

Expected: 109 / 164 events lost = fraction 0.665, exact-binomial p ≈ 3×10⁻⁵, FDR ≈ 0.003.

In [1]:
import pandas as pd
from scipy import stats
from pathlib import Path

df = pd.read_csv(Path('..') / 'data' / 'derived' / 'msk50k_within_patient_LOH_bias.csv')
print('table shape:', df.shape)
df.head()

table shape: (110, 8)


,allele,locus,n_lost,n_retained,n_total,frac_lost,p,fdr
0,B*15:10,B,15,23,38,0.394737,0.255875,0.776561
1,A*02:06,A,31,43,74,0.418919,0.200678,0.712084
2,B*35:02,B,94,129,223,0.421525,0.022584,0.272754
3,B*08:01,B,320,424,744,0.430108,0.000156,0.008563
4,B*38:02,B,13,17,30,0.433333,0.584665,0.936599


In [2]:
row = df[df.allele == 'B*58:01'].iloc[0]
k, n = int(row.n_lost), int(row.n_total)
frac = k / n
p = stats.binomtest(k, n, 0.5, alternative='two-sided').pvalue

print(f'B*58:01: {k}/{n} events lost  ->  fraction-lost = {frac:.3f}')
print(f'exact binomial p = {p:.2e}')
print(f'FDR (from table)  = {row.fdr:.4f}')

assert round(frac, 3) == 0.665, 'fraction-lost should reproduce 0.665'
assert p < 1e-4, 'should be highly significant'
print('\nReproduced: B*58:01 is preferentially lost, fraction 0.665, FDR ~0.003.')

B*58:01: 109/164 events lost  ->  fraction-lost = 0.665
exact binomial p = 2.99e-05
FDR (from table)  = 0.0033

Reproduced: B*58:01 is preferentially lost, fraction 0.665, FDR ~0.003.


## Context: it is the only individually significant *loss*-biased allele
The other FDR<0.05 alleles are preferentially *retained* — confirming the locus-wide HLA-B loss/retention hierarchy.

In [3]:
sig = df[df.fdr < 0.05].copy()
sig['direction'] = sig.frac_lost.apply(lambda x: 'LOST' if x > 0.5 else 'RETAINED')
sig.sort_values('frac_lost', ascending=False)[['allele','locus','n_lost','n_total','frac_lost','fdr','direction']]

,allele,locus,n_lost,n_total,frac_lost,fdr,direction
104,B*58:01,B,109,164,0.664634,0.003294,LOST
7,B*07:02,B,371,845,0.439053,0.016256,RETAINED
3,B*08:01,B,320,744,0.430108,0.008563,RETAINED


## Second headline: engageability is orthogonal to HED (r ≈ −0.02)
This needs the per-allele engageability score alongside HED. `data/derived/hed_per_allele.csv` carries HED; export the matching `engageability_scores.csv` from the engageability session (see `docs/EXPORT_PROMPTS.md`) into `data/derived/`, then this cell reproduces the r ≈ −0.02 orthogonality.

In [4]:
eng_path = Path('..') / 'data' / 'derived' / 'engageability_scores.csv'
if eng_path.exists():
    hed = pd.read_csv(Path('..') / 'data' / 'derived' / 'hed_per_allele.csv')
    eng = pd.read_csv(eng_path)
    m = hed.merge(eng, on='allele')
    r, p = stats.pearsonr(m['mean_HED'], m['engageability'])
    print(f'engageability vs HED: Pearson r = {r:.3f} (p={p:.3g}), n={len(m)}')
else:
    print('engageability_scores.csv not committed yet — export it from the engageability session.')

engageability_scores.csv not committed yet — export it from the engageability session.
